# 05 · Pandas 집계, 변환, 통계

빈도·평균·분산·상관관계를 직접 계산과 Pandas 코드로 비교합니다.

> 위에서 아래로 실행하세요. 예제 데이터는 노트북 안에서 만듭니다. 코드 셀 아래의 출력으로 결과를 확인하고, 실제 데이터에서는 열 이름·단위·기간을 먼저 확인하세요.

## 그룹별 건수

**코드 → 코드 개념**: `groupby`는 같은 키의 행을 묶는다. `size`는 행 수, `count`는 비결측 값 수다.

**코드 사용법**: 결측이 있는 표에서 차이를 확인한다.

In [ ]:
import pandas as pd
import numpy as np
df = pd.DataFrame({"machine": ["A", "A", "B", "B", "B"],
                   "vibration": [2.1, np.nan, 2.4, 2.9, 4.1],
                   "temperature": [72, 83, 75, 80, 85]})
print(df.groupby("machine").size())
print(df.groupby("machine")["vibration"].count())
print(df["machine"].value_counts())

**같은 결과를 얻는 방법과 선택 이유**

- `value_counts`는 한 열의 빈도를 가장 빨리 본다. 여러 집계와 합쳐야 하면 `groupby().agg()`가 좋다.
- 결측이 있으면 `size`와 `count`가 달라진다. 평균을 보고할 때 유효 표본 수도 함께 적는다.

## 그룹 요약과 여러 방법

**코드 → 코드 개념**: `agg`는 그룹별 결과를 한 행으로 축약하고 여러 통계를 동시에 구한다.

**코드 사용법**: 직접 반복과 `groupby` 결과를 비교한다.

In [ ]:
manual = {}
for name in df["machine"].unique():
    manual[name] = df.loc[df["machine"] == name, "temperature"].mean()
summary = df.groupby("machine").agg(n=("temperature", "size"),
                                     mean_temp=("temperature", "mean"),
                                     median_vibration=("vibration", "median"))
print(manual)
print(summary)
assert all(np.isclose(summary.loc[k, "mean_temp"], v) for k, v in manual.items())

**같은 결과를 얻는 방법과 선택 이유**

- 반복문은 원리를 이해하거나 그룹마다 매우 다른 처리를 할 때 좋다. 같은 계산을 모든 그룹에 적용할 때 `groupby`가 간결하고 누락 위험이 적다.
- 평균은 극단값에 민감하고 중앙값은 덜 민감하다. 둘을 함께 보면 치우침을 알 수 있다.

## agg, transform, apply

**코드 → 코드 개념**: `agg`는 그룹을 줄이고 `transform`은 원래 행 수를 유지한다.

**코드 사용법**: 각 행에 자기 설비의 평균과 편차를 붙인다.

In [ ]:
df["machine_mean"] = df.groupby("machine")["temperature"].transform("mean")
df["deviation"] = df["temperature"] - df["machine_mean"]
print(df)
print(df.groupby("machine")["temperature"].apply(lambda s: s.max() - s.min()))

**같은 결과를 얻는 방법과 선택 이유**

- 그룹 요약 표가 필요하면 `agg`, 각 행에 그룹 통계를 붙이면 `transform`을 쓴다.
- `apply`는 내장 집계로 표현하기 어려운 사용자 정의 계산에 쓴다. 단순 평균·최대에는 내장 메서드가 의도도 분명하고 보통 더 빠르다.

## 분산·표준편차·사분위

**코드 → 코드 개념**: 분산은 평균에서 떨어진 정도의 제곱 평균, 표준편차는 원래 단위의 변동성이다.

**코드 사용법**: 표본 표준편차와 분위수를 확인한다.

In [ ]:
x = df["temperature"]
print(x.mean(), x.median(), x.var(), x.std())
print(x.quantile([0.25, 0.5, 0.75]))
print(x.describe())

**같은 결과를 얻는 방법과 선택 이유**

- `describe`는 한 번에 여러 통계를 살필 때 좋고, `.quantile()`은 특정 분위수만 정확히 계산할 때 쓴다.
- Pandas 분산·표준편차 기본값은 `ddof=1`이다. NumPy 기본값과 다르므로 비교할 때 맞춘다.

## 상관계수와 해석

**코드 → 코드 개념**: Pearson 상관계수는 두 수치의 선형 동행 정도를 나타낸다.

**코드 사용법**: 결측이 있는 열 쌍의 상관을 구한다.

In [ ]:
print(df[["temperature", "vibration"]].corr())
print(df["temperature"].corr(df["vibration"]))
print(df[["temperature", "vibration"]].corr(method="spearman"))

**같은 결과를 얻는 방법과 선택 이유**

- `.corr()` 행렬은 여러 센서를 한 번에 훑기 좋고, 두 Series의 `.corr()`는 특정 관계만 볼 때 좋다.
- Pearson은 선형 관계, Spearman은 순위 기반 단조 관계를 본다. 상관은 원인을 증명하지 않으며 표본 수와 시간 추세를 함께 확인한다.

## 원본 학습 자료

[`1. lecture/02_Pandas/basic`](../1.%20lecture/02_Pandas/basic), [`1. lecture/02_Pandas/statistics`](../1.%20lecture/02_Pandas/statistics)